# 自动化测试

学习目标：能用 Node.js 内置测试运行器验证同步、异步及异常行为，控制副作用并阅读覆盖率。

前置知识：函数、模块、异常、Promise 与 async/await。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/24-testing/。

1. [mean.mjs](scripts/24-testing/mean.mjs)：纯计算与异步读取接口。
2. [mean.test.mjs](scripts/24-testing/mean.test.mjs)：正常值、边界和异常断言。
3. [async.test.mjs](scripts/24-testing/async.test.mjs)：等待拒绝、调用记录与局部 mock。
4. [isolation-a.test.mjs](scripts/24-testing/isolation-a.test.mjs)：第一份隔离测试。
5. [isolation-b.test.mjs](scripts/24-testing/isolation-b.test.mjs)：第二份隔离测试。
6. [expected-failure.mjs](scripts/24-testing/expected-failure.mjs)：独立失败示例。

Step 1：运行正常测试。

```bash
node --test scripts/24-testing/mean.test.mjs scripts/24-testing/async.test.mjs scripts/24-testing/isolation-a.test.mjs scripts/24-testing/isolation-b.test.mjs
```

Step 2：运行测试并收集覆盖率。

```bash
node --test --experimental-test-coverage scripts/24-testing/mean.test.mjs scripts/24-testing/async.test.mjs scripts/24-testing/isolation-a.test.mjs scripts/24-testing/isolation-b.test.mjs
```

Step 3：单独运行预期失败测试。

```bash
node scripts/24-testing/expected-failure.mjs
```

## 1 先定义可测试的函数边界

测试的标准应来自接口约定，而不是把实现抄一遍。本例约定 mean 的输入是有限数值数组且顺序求和不溢出；空数组返回 null，非空数组返回平均值。代码直接按这一约定计算，仍有浮点舍入误差；读取函数 loadMean 通过参数接收异步读取器，再解析 JSON 并调用计算。纯计算不访问文件、时间或全局变量，使边界值测试稳定。

JSON.parse 自身的语法错误与读取器的失败直接向外传播，供同步/异步异常断言观察；不为展示测试而额外包装输入拒绝逻辑。读取器通过依赖注入传入，即调用者决定 I/O 如何发生；真实文件读取与测试替身可以使用同一个接口。

配套 [mean.mjs](scripts/24-testing/mean.mjs)：

```javascript
export function mean(values) {
  if (values.length === 0) return null;
  const sum = values.reduce((total, value) => total + value, 0);
  return sum / values.length;
}

export async function loadMean(readText) {
  const text = await readText();
  return mean(JSON.parse(text));
}
```

## 2 同步断言、异常与边界值

node:test 注册测试，node:assert/strict 提供严格断言；它们是 Node.js 宿主模块，不属于 ECMAScript 标准。equal 在严格入口下使用严格比较，deepEqual 检查嵌套结构。throws 接收一个尚未调用的函数，检查同步异常；不能把已经抛错的表达式直接作为参数。

本例的合法边界值包括空数组、零、负数和单项数组。有限数相加也可能溢出，但本例输入约定已排除这种情况；同步异常单独用原生 JSON.parse 的语法错误演示。不要把异常都当作通过条件：至少核对错误类型或可区分的消息。测试每次创建输入，避免修改共享夹具影响后续测试。

配套 [mean.test.mjs](scripts/24-testing/mean.test.mjs)：

```javascript
import test from "node:test";
import assert from "node:assert/strict";
import { mean } from "./mean.mjs";

// 第一组同时检查返回值和输入未被修改，避免只验证计算结果。
test("mean: values and no mutation", () => {
  const values = [-2, 0, 8];
  assert.equal(mean(values), 2);
  assert.deepEqual(values, [-2, 0, 8]);
});
test("mean: empty and singleton", () => {
  assert.equal(mean([]), null);
  assert.equal(mean([0]), 0);
});
// JSON.parse 本身产生同步异常，throws 接收尚未调用的函数。
test("parse: malformed JSON", () => {
  assert.throws(() => JSON.parse("{"), SyntaxError);
});
// 多组具体输入核对同一接口性质，不重复实现求平均的算法。
test("mean: matching values", () => {
  for (const value of [4, -4, 0.5]) {
    const values = [value, value];
    assert.equal(mean(values), value);
    assert.deepEqual(values, [value, value]);
  }
});
// → 四项测试通过；测试输入与断言说明预期行为，无须依赖输出耗时。
```

## 3 异步断言与 mock 的范围

异步测试必须返回 Promise 或使用 async 并等待所有工作。assert.rejects 等待拒绝，不能用 assert.throws 检查 async 函数。测试函数结束后仍在运行的工作会让完成条件失真，并可能被运行器报告为测试外异常。

mock 是可控的测试替身或调用记录器。这里 t.mock.fn 替代读取边界，t.mock.method 临时替代对象方法；测试上下文的 mock 会在测试结束时恢复。只替换 I/O 等边界，仍执行真实的解析与计算，否则容易只证明替身返回了设定值。

配套 [async.test.mjs](scripts/24-testing/async.test.mjs)：

```javascript
import test from "node:test";
import assert from "node:assert/strict";
import { loadMean } from "./mean.mjs";

test("loadMean: injected reader", async (t) => {
  // 替换读文件边界，固定输入；调用记录用来检查依赖被调用的方式。
  const readText = t.mock.fn(async () => "[2,6]");
  assert.equal(await loadMean(readText), 4);
  assert.equal(readText.mock.callCount(), 1);
  assert.deepEqual(readText.mock.calls[0].arguments, []);
});
// 两种失败来自读取与 JSON 语法；每个 rejects 都要等待。
test("loadMean: read and parse rejection", async () => {
  await assert.rejects(loadMean(async () => { throw new Error("disk failed"); }), /disk failed/);
  await assert.rejects(loadMean(async () => "{"), SyntaxError);
});
test("loadMean: method boundary", async (t) => {
  const reader = { async read() { return "[1]"; } };
  const original = reader.read;
  t.mock.method(reader, "read", async () => "[4,8]");
  assert.equal(await loadMean(() => reader.read()), 6);
  assert.equal(reader.read.mock.callCount(), 1);
  // 在同一测试中恢复，才能直接观察原方法已经重新生效。
  t.mock.restoreAll();
  assert.equal(reader.read, original);
});
// → 三项异步测试均等待完成；拒绝原因和 mock 调用次数都须匹配。
```

## 4 文件隔离与失败诊断

Node.js --test 默认按文件启用进程隔离；同一测试文件内的测试仍可共享模块状态。--test-concurrency 控制并行测试进程数，不会把同一函数自动拆到多个线程。

以下两个文件使用相同全局属性名，但各自先断言该属性不存在，说明进程隔离保护了它们。真实代码仍应避免依赖全局污染：若改为关闭隔离，不能继续依赖这个保证。失败文件单独运行，预期退出状态为 1，并能定位 AssertionError；它不应混进正常测试入口。

配套 [isolation-a.test.mjs](scripts/24-testing/isolation-a.test.mjs)：

```javascript
import test from "node:test";
import assert from "node:assert/strict";
test("isolated A", () => {
  assert.equal(globalThis.notebookTestMarker, undefined);
  globalThis.notebookTestMarker = "A";
});
```

配套 [isolation-b.test.mjs](scripts/24-testing/isolation-b.test.mjs)：

```javascript
import test from "node:test";
import assert from "node:assert/strict";
test("isolated B", () => {
  assert.equal(globalThis.notebookTestMarker, undefined);
  globalThis.notebookTestMarker = "B";
});
```

配套 [expected-failure.mjs](scripts/24-testing/expected-failure.mjs)：

```javascript
import test from "node:test";
import assert from "node:assert/strict";
test("deliberate mismatch", () => {
  assert.equal(2 + 2, 5); // → AssertionError，运行器报告 fail 1，进程退出 1
});
```

## 5 覆盖率与持续执行

覆盖率（coverage）记录哪些语句、分支和函数在本次执行中被触达。Node.js 24.11.0 的 --experimental-test-coverage 仍标为实验功能；开启后终端会给出统计。覆盖率高不能证明断言有效，也不表示所有业务输入正确。

这里把覆盖率纳入同一组实际测试，观察空数组和普通计算分支是否被触达。修改代码后重新运行，而不是沿用旧统计。调试时可执行单个文件，持续集成时选择明确的正常文件列表，避免独立反例被自动收集。

## 本章小结

- 测试用明确输入与断言验证接口行为，异步完成与拒绝也属于接口。
- 隔离和 mock 帮助控制副作用，不能替代真实逻辑测试。
- 覆盖率说明触达情况；失败原因与断言质量仍需阅读判断。

## 练习

1. 在 matching values 测试中新增 8 和 -8 两组值，保留求和不溢出的输入条件；可核对标准：平均值分别为 8、-8，输入数组未改变。
2. 把读取器改成先等待一次 Promise 再返回文本；标准：loadMean 测试仍实际等待结果，调用次数为 1。
3. 故意破坏空数组分支并运行测试；标准：对应测试失败并指出 null 与实际值差异，恢复实现后再次全部通过。

### 提示

1. 只扩充测试数据，不增加被测函数入口校验。
2. 使用 async 替身，并 await loadMean。
3. 只临时移除空数组返回 null 的分支，观察失败后恢复。


### 参考解析

1. [8,8] 与 [-8,-8] 均满足条件，平均值为 8 与 -8；deepEqual 同时确认输入未修改。
2. `t.mock.fn(async () => { await Promise.resolve(); return "[2,6]"; })` 返回异步结果；await loadMean 后仍得到 4、调用次数为 1。
3. 临时删除该分支后，空数组会执行 0 / 0，实际得到 NaN；与期望 null 的断言失败。恢复分支后重新运行正常文件列表，确认没有把故意破坏留在交付代码里。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| Node.js 24.11.0 | [node:test](https://nodejs.org/download/release/v24.11.0/docs/api/test.html)，Test runner execution model、Collecting code coverage、Mocking、Extraneous asynchronous activity、MockTracker；[node:assert](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html)，strict mode、deepStrictEqual、throws 与 rejects。 |
| TC39（ECMA-262 第 16 版） | [§25.5 JSON](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-json-object)；[§6.1.6.1.7 Number::add](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-numeric-types-number-add)：解析异常、数据转换及有限数值相加时的溢出边界。 |
